In [1]:
import setup
setup.init_django()

In [2]:
import requests

from datetime import datetime
from urllib.parse import urlencode

from django.conf import settings
from pytz import timezone

ALPHA_VANTAGE_API_KEY = settings.ALPHA_VANTAGE_API_KEY
assert ALPHA_VANTAGE_API_KEY != ""

In [3]:
from market.models import Company, StockQuote

In [4]:
symbol="AAPL"

company, created = Company.objects.update_or_create(
    ticker=symbol,
    defaults = {
        'name': 'Apple Inc.',
        'active': True
    }
)

In [5]:
company.ticker, company.id

('AAPL', 1)

In [6]:
month = datetime.now().strftime("%Y-%m")
month

'2024-10'

In [7]:
params = {
    "function": "TIME_SERIES_INTRADAY",
    "apikey": ALPHA_VANTAGE_API_KEY,
    "symbol": symbol,
    "interval": "1min",
    "extended_hours": "false",
    "outputsize": "full",
    "month": month,
}

In [8]:
params = urlencode(params)

In [9]:
url = f'https://www.alphavantage.co/query?{params}'
r = requests.get(url)
data = r.json()

In [10]:
data

{'Meta Data': {'1. Information': 'Intraday (1min) open, high, low, close prices and volume',
  '2. Symbol': 'AAPL',
  '3. Last Refreshed': '2024-10-15 15:59:00',
  '4. Interval': '1min',
  '5. Output Size': 'Full size',
  '6. Time Zone': 'US/Eastern'},
 'Time Series (1min)': {'2024-10-15 15:59:00': {'1. open': '233.7599',
   '2. high': '234.0000',
   '3. low': '233.7200',
   '4. close': '233.8000',
   '5. volume': '1454409'},
  '2024-10-15 15:58:00': {'1. open': '233.6258',
   '2. high': '233.8200',
   '3. low': '233.6200',
   '4. close': '233.7600',
   '5. volume': '409333'},
  '2024-10-15 15:57:00': {'1. open': '233.7100',
   '2. high': '233.7750',
   '3. low': '233.5000',
   '4. close': '233.6200',
   '5. volume': '287917'},
  '2024-10-15 15:56:00': {'1. open': '233.6350',
   '2. high': '233.7900',
   '3. low': '233.5700',
   '4. close': '233.7050',
   '5. volume': '218782'},
  '2024-10-15 15:55:00': {'1. open': '233.8700',
   '2. high': '233.8750',
   '3. low': '233.4620',
   '4. c

In [11]:
stock_market_data_mapping = {
    '1. open': 'open_price',
    '2. high': 'high',
    '3. low': 'low',
    '4. close': 'close_price', 
    '5. volume': 'volume'
}

def rename_sm_keys(og_dict):
    return {stock_market_data_mapping[k]: v for k,v in og_dict.items()}

In [12]:
tz = timezone("US/Eastern")

def timezone_as_datetime_obj(timezone_str, use_tz=True):
    as_datetime = datetime.strptime(timezone_str, "%Y-%m-%d %H:%M:%S")
    if not use_tz:
        return as_datetime
    return tz.localize(as_datetime)

In [13]:
dataset = data['Time Series (1min)']
timestamps = dataset.keys()

cleaned_dataset = []
for timestamp in timestamps:
    sm_data_raw = dataset[timestamp]
    cleaned_data = rename_sm_keys(sm_data_raw)
    cleaned_data['time'] = timezone_as_datetime_obj(timestamp, use_tz=True)
    cleaned_data['company'] = company
    cleaned_dataset.append(cleaned_data)

In [14]:
len(cleaned_dataset)

4290

In [22]:
batch_size = 500
for i in range(0, len(cleaned_dataset), batch_size):
    batch_chunk = cleaned_dataset[i:i+batch_size]
    new_data = [StockQuote(**x) for x in batch_chunk]
    StockQuote.objects.bulk_create(new_data, ignore_conflicts=True)

In [23]:
# StockQuote.objects.all().delete()

In [24]:
qs = StockQuote.objects.all()
# for obj in qs:
#     print(obj.time, obj.open_price)

In [25]:
qs.count()

4290